# astream과 응답 스트림

실제 LangGraph `astream`이 한 번의 최종 응답을 여러 사용자 정의 조각으로 전달하는지 관찰합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

## 코드 읽기와 개념 경계

`agent.astream(input, stream_mode='custom')`은 최종 사전 하나가 아니라 비동기 반복 가능한 스트림을 반환합니다. `async for`는 새 조각이 도착할 때마다 반복 본문을 실행하며, 이 예제는 각 조각의 `token`을 목록에 추가합니다. `''.join(chunks).strip()`은 조각을 연결해 최종 문장을 만들지만, 연결 전 목록도 출력하여 경계가 어디였는지 확인합니다. 스트림 조각 수는 답의 의미 단위나 모델 토큰 수와 같다고 가정하지 않습니다.

파일 다운로드처럼 이미 만들어진 큰 바이트를 나누는 스트림과 Agent가 생성하면서 내보내는 스트림은 지연 원인이 다릅니다. 첫 조각까지의 시간, 조각 사이의 시간, 전체 완료 시간을 따로 측정해야 합니다. 또한 소비자가 연결을 끊었을 때 생산 작업을 취소하지 않으면 화면은 닫혔는데 모델 호출은 계속될 수 있습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 스트림 수집

`stream_mode='custom'`은 노드가 기록한 토큰 조각을 받습니다.

In [2]:
from agent_service.agent import agent
chunks = []
async for chunk in agent.astream(
    {'messages': ['세 단어 질문']}, stream_mode='custom'
):
    chunks.append(chunk['token'])
print(chunks)
assert len(chunks) >= 4

['질문을 ', '확인했습니다: ', '세 ', '단어 ', '질문 ']


## 중간 결과 관찰

마지막 조각을 기다리지 않아도 첫 조각을 먼저 처리할 수 있습니다.

In [3]:
first = chunks[0]
joined = ''.join(chunks).strip()
print('첫 조각:', first)
print('전체:', joined)
assert joined == '질문을 확인했습니다: 세 단어 질문'

첫 조각: 질문을 
전체: 질문을 확인했습니다: 세 단어 질문


## 반복 실험과 결과 해석

질문의 단어 수만 바꾸고 생성된 조각 수를 비교합니다.

In [4]:
counts = {}
for question in ['짧음', '두 단어', '세 개의 단어']:
    count = 0
    async for _ in agent.astream({'messages': [question]}, stream_mode='custom'):
        count += 1
    counts[question] = count
print(counts)
assert counts['짧음'] < counts['세 개의 단어']

{'짧음': 3, '두 단어': 4, '세 개의 단어': 5}
